# Real-data Corridor-Width Cluster Vecchia (4x4 lag432)

This notebook fits `Lag432CorridorVecchia`, the maintained fixed-longitude 4/3/2 corridor model, on one day of July 2024 real data.

Design:
- clusters are built on regular `Latitude/Longitude` grid cells with fixed `4x4` blocks;
- same-time conditioning uses 4 previous clusters in max-min order;
- lagged conditioning uses corridor-width blocks: 3 at `t-1`, 2 at `t-2`;
- default one-step reference displacement is `delta=0.126`, giving `t-1=[0.063, 0.189]` and `t-2=[0, 0.252]` longitude corridors;
- covariance uses the source coordinates in `input_map`; target-block geometry uses the regular `Latitude/Longitude` grid.

This 432 version is the lightweight CPU/GPU sanity-check model. Use the 643 wrapper for the safer final real-data candidate.


In [1]:
import gc
import logging
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.nn import Parameter

sys.path.insert(0, '/Users/joonwonlee/Documents/GEMS_TCO-1/src')

from GEMS_TCO.data import ProcessedDataLoader
from GEMS_TCO.vecchia.corridor_neighbors.corridor_lag432 import (
    BLOCK_SHAPE,
    SPEC_NAME as VECCHIA_SPEC_NAME,
    Lag432CorridorVecchia,
    model_spec as corridor_width_432_spec,
)

logging.basicConfig(level=logging.INFO, format='%(levelname)s:%(name)s:%(message)s', force=True)

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())

ROUND_DECIMALS = 5

def round_numeric_df(df, digits=ROUND_DECIMALS):
    out = df.copy()
    num_cols = out.select_dtypes(include=[np.number]).columns
    out[num_cols] = out[num_cols].round(digits)
    return out

def round_numeric_series(s, digits=ROUND_DECIMALS):
    out = s.copy()
    for idx, val in out.items():
        if isinstance(val, (float, np.floating)) and np.isfinite(val):
            out[idx] = round(float(val), digits)
    return out


torch: 2.5.1
cuda available: False


In [2]:
YEAR = '2024'
MONTH = 7
DAYS_LIST = [13]  # 0-based day index. July 14 here. Full July: list(range(31))
LAT_RANGE = [-3, 2]
LON_RANGE = [121, 131]
DATA_DIR = Path('/Users/joonwonlee/Documents/GEMS_DATA')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE = torch.double
SMOOTH = 0.5
SECOND_LAG_STRIDE = 2

# Fixed real-data corridor-width 4x4 lag432 model.
REFERENCE_ADVEC_LON_ABS = 0.126
MODEL_SPEC = corridor_width_432_spec(REFERENCE_ADVEC_LON_ABS)
TARGET_CHUNK_SIZE = 96  # reduce to 32/64 if GPU memory is tight
COVARIANCE_BACKEND = 'auto'  # fused native CPU when built; Torch fallback otherwise

LBFGS_LR = 1.0
LBFGS_MAX_STEPS = 5
LBFGS_MAX_EVAL = 20
LBFGS_HISTORY_SIZE = 10
LBFGS_TOLERANCE_GRAD = 1e-5  # retain the stricter inner L-BFGS stopping rule
GRAD_TOL = 1e-4  # outer stopping rule; accepted speed/accuracy trade-off
INIT_PHYSICAL = {
    'signal_variance': 10.0,
    'range_lat': 0.30,
    'range_lon': 0.40,
    'range_time': 2.0,
    'advec_lat': 0.05,
    'advec_lon': -0.10,
    'nugget': 2.5,
}

OUT_DIR = Path('/Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/log')
OUT_PREFIX = 'real_vecc_2024_corridor_width_4x4_lag432_092126'
SAVE_CSV = True

print('device:', DEVICE)
print('covariance backend:', COVARIANCE_BACKEND)
print('days:', DAYS_LIST)
print('spec:', VECCHIA_SPEC_NAME)
print('model spec:', MODEL_SPEC)


device: cpu
covariance backend: auto
days: [13]
spec: corridor_width_4x4_lag432
model spec: {'spec_name': 'corridor_width_4x4_lag432', 'conditioning_mode': 'fixed_longitude_corridor_width', 'block_shape': (4, 4), 'lag_counts': (4, 3, 2), 'lag_pattern': '4/3/2', 'reference_advec_lon_abs': 0.126, 'lag1_lon_offset': 0.126, 'lag2_lon_offset': 0.252, 'lag1_lon_interval': (0.063, 0.189), 'lag2_lon_interval': (0.0, 0.252), 'corridor_anchor_mode': 'width'}


## Load Real Data

Important: this model does **not** need point-level max-min ordering or a point `nns_map`. `ProcessedDataLoader` therefore loads the monthly grid without computing a point ordering. The cluster model builds its own cluster max-min ordering internally.

In [3]:
data_load_instance = ProcessedDataLoader(DATA_DIR)

df_map, _, _, monthly_mean = data_load_instance.load_monthly_grids(
    years=[YEAR],
    months=[MONTH],
    latitude_range=LAT_RANGE,
    longitude_range=LON_RANGE,
    compute_ordering=False,
)

key_idx = sorted(df_map)
print('n hourly slots:', len(key_idx))
print('monthly_mean:', monthly_mean)
print('first key:', key_idx[0], 'last key:', key_idx[-1])

base_grid_coords_np = df_map[key_idx[0]][['Latitude', 'Longitude']].to_numpy(dtype=np.float64)
print('base_grid_coords_np:', base_grid_coords_np.shape)
print('n unique lat/lon:', len(np.unique(base_grid_coords_np[:,0])), len(np.unique(base_grid_coords_np[:,1])))

INFO:GEMS_TCO.data.loading:Pooled mean over loaded ColumnAmountO3 values: 257.9726


n hourly slots: 248
monthly_mean: 257.97261042523166
first key: 2024_07_y24m07day01_hm00:53 last key: 2024_07_y24m07day31_hm07:48
base_grid_coords_np: (18126, 2)
n unique lat/lon: 114 159


In [4]:
def assert_grid_order_consistent(keys, base_coords):
    for k in keys:
        coords = df_map[k][['Latitude', 'Longitude']].to_numpy(dtype=np.float64)
        if coords.shape != base_coords.shape or not np.allclose(coords, base_coords, equal_nan=True):
            raise RuntimeError(f'Grid coordinate order differs at {k}; cluster local-index mapping is not reusable.')

for day_idx in DAYS_LIST:
    hour_indices = [day_idx * 8, (day_idx + 1) * 8]
    assert_grid_order_consistent(key_idx[hour_indices[0]:hour_indices[1]], base_grid_coords_np)
print('grid order consistency check passed for selected days')

grid order consistency check passed for selected days


In [5]:
daily_hourly_maps = {}
selected_key_map = {}

for day_idx in DAYS_LIST:
    hour_indices = [day_idx * 8, (day_idx + 1) * 8]
    selected_keys = key_idx[hour_indices[0]:hour_indices[1]]
    if len(selected_keys) != 8:
        raise RuntimeError(f'day_idx={day_idx} does not contain eight hourly slots')
    selected_key_map[day_idx] = selected_keys
    day_frames = {key: df_map[key] for key in selected_keys}
    day_hourly_map, _ = data_load_instance.build_model_tensors(
        day_frames,
        ozone_mean=monthly_mean,
        time_slice=(0, 8),
        spatial_order=None,  # cluster model constructs the block ordering
        dtype=DTYPE,
        use_source_coordinates=True,
    )
    daily_hourly_maps[day_idx] = {k: v.to(DEVICE) for k, v in day_hourly_map.items()}

rows = []
for day_idx, maps in daily_hourly_maps.items():
    n_valid = sum(int((~torch.isnan(v[:, 2])).sum().item()) for v in maps.values())
    n_total = sum(int(v.shape[0]) for v in maps.values())
    rows.append({
        'day_idx': day_idx,
        'day': f'{YEAR}-{MONTH:02d}-{day_idx + 1:02d}',
        'n_time_slots': len(maps),
        'n_rows_total': n_total,
        'n_valid_o3': n_valid,
        'valid_rate': n_valid / n_total,
        'first_slot': selected_key_map[day_idx][0],
        'last_slot': selected_key_map[day_idx][-1],
    })
load_summary = pd.DataFrame(rows)
display(round_numeric_df(load_summary))

,day_idx,day,n_time_slots,n_rows_total,n_valid_o3,valid_rate,first_slot,last_slot
0,13,2024-07-14,8,145008,144078,0.99359,2024_07_y24m07day14_hm00:53,2024_07_y24m07day14_hm07:48


## Parameter Helpers

In [6]:
P_LABELS = ['signal_variance', 'range_lat', 'range_lon', 'range_time', 'advec_lat', 'advec_lon', 'nugget']

def physical_to_log_phi(params):
    signal_variance = float(params['signal_variance'])
    range_lat = float(params['range_lat'])
    range_lon = float(params['range_lon'])
    range_time = float(params['range_time'])
    nugget = float(params['nugget'])
    phi2 = 1.0 / range_lon
    phi1 = signal_variance * phi2
    phi3 = (range_lon / range_lat) ** 2
    phi4 = (range_lon / range_time) ** 2
    return [
        np.log(phi1), np.log(phi2), np.log(phi3), np.log(phi4),
        float(params['advec_lat']), float(params['advec_lon']), np.log(nugget),
    ]

def make_params_list(init_physical=INIT_PHYSICAL):
    initial_vals = physical_to_log_phi(init_physical)
    return [Parameter(torch.tensor([val], dtype=DTYPE, device=DEVICE)) for val in initial_vals]

print('init log phi:', np.round(physical_to_log_phi(INIT_PHYSICAL), 4))

init log phi: [ 3.2189  0.9163  0.5754 -3.2189  0.05   -0.1     0.9163]


## Fit One Day

In [7]:
def instantiate_model(day_map):
    return Lag432CorridorVecchia(
        smooth=SMOOTH,
        input_map=day_map,
        grid_coords=base_grid_coords_np,
        reference_advec_lon_abs=REFERENCE_ADVEC_LON_ABS,
        second_lag_stride=SECOND_LAG_STRIDE,
        target_chunk_size=TARGET_CHUNK_SIZE,
        covariance_backend=COVARIANCE_BACKEND,
    )

def fit_one(day_idx):
    day_map = daily_hourly_maps[day_idx]
    params_list = make_params_list()
    model = instantiate_model(day_map)

    t0 = time.time()
    model.precompute_conditioning_sets()
    precompute_s = time.time() - t0

    optimizer = model.make_lbfgs_optimizer(
        params_list,
        lr=LBFGS_LR,
        max_iter=LBFGS_MAX_EVAL,
        max_eval=LBFGS_MAX_EVAL,
        tolerance_grad=LBFGS_TOLERANCE_GRAD,
        history_size=LBFGS_HISTORY_SIZE,
    )

    t1 = time.time()
    fit_result = model.fit_lbfgs(
        params_list, optimizer, max_steps=LBFGS_MAX_STEPS, grad_tol=GRAD_TOL
    )
    fit_s = time.time() - t1

    est = fit_result.interpretable_parameters
    row = {
        'day_idx': day_idx,
        'day': f'{YEAR}-{MONTH:02d}-{day_idx + 1:02d}',
        'smooth': SMOOTH,
        'spec_name': VECCHIA_SPEC_NAME,
        'block_shape': f'{BLOCK_SHAPE[0]}x{BLOCK_SHAPE[1]}',
        'lag_pattern': MODEL_SPEC['lag_pattern'],
        'reference_advec_lon_abs': REFERENCE_ADVEC_LON_ABS,
        'negative_log_likelihood': fit_result.final_nll,
        'optimization_steps': fit_result.steps_completed,
        'converged': fit_result.converged,
        'max_abs_gradient': fit_result.max_abs_gradient,
        'objective_evaluations': fit_result.objective_evaluations,
        'objective_cache_hits': fit_result.cache_hits,
        'precompute_s': round(precompute_s, ROUND_DECIMALS),
        'fit_s': round(fit_s, ROUND_DECIMALS),
        **{f'estimate_{k}': float(est[k]) for k in P_LABELS},
        **model.cluster_summary(),
    }

    del model, params_list, optimizer
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    return row

results = []
for day_idx in DAYS_LIST:
    print(f'\n=== fitting day_idx={day_idx} ({YEAR}-{MONTH:02d}-{day_idx+1:02d}) ===')
    row = fit_one(day_idx)
    results.append(row)
    print(round_numeric_series(pd.Series(row)).to_string())

results_df = pd.DataFrame(results)
display(round_numeric_df(results_df))

INFO:GEMS_TCO.vecchia.grouped_batched:Pre-computing corridor Vecchia (smooth=0.5, block=(4, 4), origin=0/0, lag_blocks=4/3/2, offsets=0.1260/0.2520, corridors=(0.063, 0.189)/(0.0, 0.252), anchor_mode=width)...



=== fitting day_idx=13 (2024-07-14) ===


INFO:GEMS_TCO.vecchia.grouped_batched:Done. clusters=1160, max_points/block=16, target_blocks=9278, target_points=144078, batches=[A:m64:b6x1, A:m64:b8x53, A:m64:b11x2, A:m64:b12x12, A:m64:b16x1092, AB:m112:b2x1, AB:m112:b7x1, AB:m112:b8x50, AB:m112:b11x2, AB:m112:b12x14, ... (26 batches)]


INFO:GEMS_TCO.vecchia._base:--- Starting Batched L-BFGS Optimization (cpu) ---


INFO:GEMS_TCO.vecchia._base:L-BFGS step 1/5: nll=1.0695009, valid=True, max_abs_gradient=4.104e-05


INFO:GEMS_TCO.vecchia._base:Final interpretable parameters: {'signal_variance': 7.783842769751583, 'range_lon': 0.28201252779027136, 'range_lat': 0.23095484967077826, 'range_time': 1.4221863413560862, 'advec_lat': 0.005111393441198041, 'advec_lon': -0.03565802619706582, 'nugget': 1.123949911425433}


INFO:GEMS_TCO.vecchia._base:Vecchia objective evaluations: 20 computed, 3 exact-state cache hits


day_idx                                                 13
day                                             2024-07-14
smooth                                                 0.5
spec_name                        corridor_width_4x4_lag432
block_shape                                            4x4
lag_pattern                                          4/3/2
reference_advec_lon_abs                              0.126
negative_log_likelihood                             1.0695
optimization_steps                                       1
converged                                             True
max_abs_gradient                                   0.00004
objective_evaluations                                   20
objective_cache_hits                                     3
precompute_s                                       0.76208
fit_s                                             60.06414
estimate_signal_variance                           7.78384
estimate_range_lat                                 0.230

,day_idx,day,smooth,spec_name,block_shape,lag_pattern,reference_advec_lon_abs,negative_log_likelihood,optimization_steps,converged,...,lag2_lon_offset,lag1_lon_interval_lo,lag1_lon_interval_hi,lag2_lon_interval_lo,lag2_lon_interval_hi,corridor_anchor_mode,grid_lon_step,corridor_block_lon_width,block_row_offset,block_col_offset
0,13,2024-07-14,0.5,corridor_width_4x4_lag432,4x4,4/3/2,0.126,1.0695,1,True,...,0.252,0.063,0.189,0.0,0.252,width,0.063,0.252,0,0


In [8]:
if SAVE_CSV:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    out_path = OUT_DIR / f'{OUT_PREFIX}_days_{min(DAYS_LIST)+1:02d}_{max(DAYS_LIST)+1:02d}.csv'
    results_df.to_csv(out_path, index=False, float_format="%.12g")
    print('saved:', out_path)

saved: /Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/log/real_vecc_2024_corridor_width_4x4_lag432_092126_days_14_14.csv
